# Breast-Lesions-USG preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.breast_lesion_usg import BreastLesionUSG, raw_imgs_path, csv_save_path
from utils.preprocessing import birads_assessment, get_value, read_breast_image

ds = BreastLesionUSG()
ds.set_dataset_name("breast-lesions-usg")
ds.lesions_usg_df.head()

### Step 1 — drop columns we cannot use (post-biopsy / not needed)

In [ ]:
ds.lesions_usg_df = ds.lesions_usg_df.drop(columns=["mask other filename", "pixel size", "verification"])
ds.lesions_usg_df.columns.tolist()

### Step 2 — map raw BI-RADS text to harmonized labels

In [ ]:
ds.lesions_usg_df["birads"] = ds.lesions_usg_df["birads"].apply(lambda x: get_value(x, birads_assessment))
ds.lesions_usg_df["birads"].value_counts(dropna=False)

### Step 3 — normalize 'not applicable' / 'not available' placeholders to `None`

In [ ]:
ds.lesions_usg_df = ds.lesions_usg_df.replace("not applicable", None)
ds.lesions_usg_df = ds.lesions_usg_df.replace("not available", None)
ds.lesions_usg_df.isna().sum()

## Preview the source image and mask for one exam

In [ ]:
import os
import matplotlib.pyplot as plt

row = ds.lesions_usg_df.iloc[0]
img = read_breast_image(os.path.join(raw_imgs_path, row["image filename"]))
plt.imshow(img, cmap="gray")
plt.axis("off")
plt.show()

## Build one exam record

In [ ]:
exam = ds.process_row(row)
exam

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"rows before per-exam processing: {len(ds.lesions_usg_df)} | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))